# Table 2: g(t)=sin(t), n=400

This notebook is a **standalone end-to-end reproduction** of Table 2.
It contains the data-generating process or the frozen raw RHC data, preprocessing,
estimators, uncertainty calculations, manuscript values, and table formatting.
It does not import any project-local module and does not read cached results.

Run all cells in a fresh Python kernel. The last cell prints only the freshly
computed corrected result and its integrity checks.

In [1]:
"""Shared, reproducible implementation of the synthetic experiments.

The implementation follows Sections 4.2--4.3 and Appendix H.1 of the paper:
outcome and action bridges are learned from their conditional-moment losses,
the stabilized losses use lambda=1 and gamma=5, and the ablation changes only
the treatment shift in U (kappa_a=0).
"""

from __future__ import annotations

import argparse
import json
import os
import random
from dataclasses import asdict, dataclass
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


torch.set_num_threads(max(1, int(os.getenv("MINIMAX_TORCH_THREADS", "1"))))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass


@dataclass(frozen=True)
class SyntheticConfig:
    n: int
    transform: str
    stabilized: bool
    repetitions: int = 1000
    dimension: int = 60
    seed: int = 20260113
    learning_rate: float = 2e-4
    momentum: float = 0.95
    # The original notebooks use 20 epochs with 40 minibatches per epoch.
    updates: int = 800
    lambda_stab: float = 1.0
    gamma_stab: float = 5.0
    # The no-confounding ablation has a much weaker action-bridge signal.  A
    # larger effective q batch and an explicit weight on the normalization
    # moment prevent the minibatch optimizer from ignoring E[I(A=1)q-1]=0.
    # This moment is already implied by the affine linear critic, so the
    # calibration changes only finite-sample optimization, not the bridge
    # equation or its population solution.
    q_ablation_batch_divisor: int = 10
    q_ablation_learning_rate_multiplier: float = 0.1
    q_ablation_normalization_weight_nonsta: float = 10.0
    q_ablation_normalization_weight_sta: float = 0.3


class BridgeNet(nn.Module):
    def __init__(self, d: int, positive: bool = False):
        super().__init__()
        self.fc1 = nn.Linear(2 * d, 2 * d)
        self.fc2 = nn.Linear(2 * d, d)
        self.fc3 = nn.Linear(d, 1)
        self.positive = positive
        if positive:
            # Start near the marginal inverse treatment probability instead
            # of the nearly-flat tail of Softplus.
            nn.init.constant_(self.fc3.bias, float(np.log(np.expm1(2.5))))

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        value = F.relu(self.fc1(value))
        value = F.relu(self.fc2(value))
        value = self.fc3(value)
        if self.positive:
            value = F.softplus(value) + 1e-4
        return value


class PropensityNet(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.fc1 = nn.Linear(d, 2 * d)
        self.fc2 = nn.Linear(2 * d, d)
        self.fc3 = nn.Linear(d, 1)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        value = F.relu(self.fc1(value))
        value = F.relu(self.fc2(value))
        return torch.sigmoid(self.fc3(value))


class OutcomeNet(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.fc1 = nn.Linear(d + 1, 2 * d)
        self.fc2 = nn.Linear(2 * d, d)
        self.fc3 = nn.Linear(d, 1)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        value = F.relu(self.fc1(value))
        value = F.relu(self.fc2(value))
        return self.fc3(value)


class ScaledOutcome(nn.Module):
    """Map a network trained on standardized outcomes back to the Y scale."""

    def __init__(self, model: nn.Module, mean: torch.Tensor, scale: torch.Tensor):
        super().__init__()
        self.model = model
        self.register_buffer("mean", mean.detach().clone())
        self.register_buffer("scale", scale.detach().clone())

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return self.mean + self.scale * self.model(value)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


@lru_cache(maxsize=None)
def transform_matrix(d: int) -> np.ndarray:
    matrix = np.full((d, d), 1.0 / (4.0 * d), dtype=np.float64)
    np.fill_diagonal(matrix, 1.0)
    return matrix


def nonlinear_transform(value: np.ndarray, matrix: np.ndarray, name: str) -> np.ndarray:
    projected = value @ matrix.T
    if name == "sin":
        return np.sin(projected)
    if name == "power":
        return projected**3
    raise ValueError(f"Unknown transform: {name}")


@lru_cache(maxsize=None)
def covariance_components(d: int):
    sigma_uz = np.full((2 * d, 2 * d), 0.1, dtype=np.float64)
    np.fill_diagonal(sigma_uz, 0.2)
    sigma_uz_inv = np.linalg.inv(sigma_uz)
    precision_uz = sigma_uz_inv[d:, :d]
    sigma_wu = np.full((d, d), 0.1, dtype=np.float64)
    sigma_wz = -sigma_wu @ precision_uz @ np.linalg.inv(sigma_uz_inv[:d, :d])

    sigma = np.full((3 * d, 3 * d), 0.1, dtype=np.float64)
    sigma[d:, d:] = sigma_uz
    sigma[:d, d : 2 * d] = sigma_wz
    sigma[d : 2 * d, :d] = sigma_wz.T
    sigma[:d, 2 * d :] = sigma_wu
    sigma[2 * d :, :d] = sigma_wu.T
    np.fill_diagonal(sigma[:d, :d], 0.2)
    sigma = (sigma + sigma.T) / 2.0
    eigval, eigvec = np.linalg.eigh(sigma)
    chol = eigvec @ np.diag(np.sqrt(np.maximum(eigval, 1e-10)))
    sigma_sigma = sigma[:d, d:] @ sigma_uz_inv
    return sigma, chol, sigma_sigma


def population_truth(d: int, kappa_a: float, sigma_sigma: np.ndarray) -> float:
    # sum(X') ~ N(0, 0.5 d); Gauss-Hermite accurately integrates E[p(X')].
    nodes, weights = np.polynomial.hermite.hermgauss(80)
    sums = np.sqrt(d) * nodes  # sqrt(2 * 0.5 d) * nodes
    propensity = 1.0 / (1.0 + np.exp(0.5 - 0.05 * sums))
    mean_a = float(np.sum(weights * propensity) / np.sqrt(np.pi))
    c_u = float(np.sum(sigma_sigma[:, d:] @ np.ones(d)))
    return 1.0 + 2.0 * d * 0.2 + kappa_a * (d + c_u) * mean_a


def generate_data(config: SyntheticConfig, rng: np.random.Generator, ablation: bool):
    d, n = config.dimension, config.n
    _, chol, sigma_sigma = covariance_components(d)
    kappa_a = 0.0 if ablation else 1.0
    x_raw = rng.normal(0.0, np.sqrt(0.5), size=(n, d))
    propensity = 1.0 / (1.0 + np.exp(0.5 - 0.05 * x_raw.sum(axis=1)))
    action = rng.binomial(1, propensity, size=n).astype(np.float64)[:, None]

    alpha_a = np.ones(d)
    kappa_vec = np.full(d, kappa_a)
    mu_a = sigma_sigma[:, :d] @ alpha_a + sigma_sigma[:, d:] @ kappa_vec
    mean_w = 0.2 + x_raw + action * mu_a
    mean_z = 0.2 + x_raw + action
    mean_u = 0.2 + x_raw + kappa_a * action
    noise = rng.normal(size=(n, 3 * d)) @ chol.T
    raw = np.concatenate([mean_w, mean_z, mean_u], axis=1) + noise
    w_raw, z_raw, u = raw[:, :d], raw[:, d : 2 * d], raw[:, 2 * d :]
    outcome = (
        action[:, 0]
        + x_raw.sum(axis=1)
        + u.sum(axis=1)
        + w_raw.sum(axis=1)
        + rng.normal(size=n)
    )[:, None]
    matrix = transform_matrix(d)
    x = nonlinear_transform(x_raw, matrix, config.transform)
    z = nonlinear_transform(z_raw, matrix, config.transform)
    w = nonlinear_transform(w_raw, matrix, config.transform)
    truth = population_truth(d, kappa_a, sigma_sigma)
    arrays = [x, z, w, action, outcome]
    tensors = [torch.from_numpy(v.astype(np.float32)) for v in arrays]
    return (*tensors, truth)


def rbf_gram(value: torch.Tensor) -> torch.Tensor:
    distance_sq = torch.cdist(value, value).square()
    with torch.no_grad():
        upper = distance_sq[torch.triu(torch.ones_like(distance_sq, dtype=torch.bool), diagonal=1)]
        positive = upper[upper > 0]
        median = torch.median(positive) if positive.numel() else value.new_tensor(1.0)
        scale = torch.clamp(median, min=1e-8)
    return torch.exp(-distance_sq / (2.0 * scale))


def conditional_moment_loss(
    residual: torch.Tensor,
    critic_input: torch.Tensor,
    kernel: str,
    stabilized: bool,
    lambda_stab: float,
    gamma_stab: float,
) -> torch.Tensor:
    if kernel == "rbf":
        gram = rbf_gram(critic_input)
    elif kernel == "linear":
        # Use the affine linear kernel.  The constant feature is essential for
        # the normalization moment E[I(A=a)q-1]=0.
        features = torch.cat([torch.ones_like(critic_input[:, :1]), critic_input], dim=1)
        gram = features @ features.T
    else:
        raise ValueError(kernel)
    if stabilized:
        identity = torch.eye(gram.shape[0], dtype=gram.dtype, device=gram.device)
        weighted = gram @ torch.linalg.solve(gamma_stab * identity + lambda_stab * gram, residual)
        return (residual.T @ weighted).squeeze() / residual.shape[0] ** 2
    return (residual.T @ gram @ residual).squeeze() / residual.shape[0] ** 2


def fit_bridge(
    config: SyntheticConfig,
    x: torch.Tensor,
    z: torch.Tensor,
    w: torch.Tensor,
    a: torch.Tensor,
    y: torch.Tensor,
    kind: str,
    ablation: bool = False,
) -> nn.Module:
    d, n = config.dimension, x.shape[0]
    positive = kind == "q"
    model = BridgeNet(d, positive=positive)
    learning_rate = config.learning_rate
    if kind == "h" and config.transform == "power":
        # The t^3 coordinates make the outcome-bridge gradient much steeper.
        learning_rate /= 10.0
    if kind == "q" and config.stabilized:
        # The preconditioned action-bridge loss is much steeper for t^3 inputs;
        # the smaller step prevents Softplus from entering its flat tail.
        learning_rate /= 10.0
    if kind == "q" and ablation and not config.stabilized:
        learning_rate *= config.q_ablation_learning_rate_multiplier
    optimizer = torch.optim.RMSprop(
        model.parameters(), lr=learning_rate, momentum=config.momentum
    )
    batch_divisor = (
        config.q_ablation_batch_divisor if kind == "q" and ablation else 40
    )
    batch = max(8, n // batch_divisor)
    for _ in range(config.updates):
        index = torch.randint(0, n, (batch,))
        xb, zb, wb, ab, yb = x[index], z[index], w[index], a[index], y[index]
        if kind == "h":
            prediction = model(torch.cat([wb, xb], dim=1))
            residual = (ab == 1).float() * (yb - prediction)
            critic = torch.cat([zb, xb], dim=1)
            kernel = "rbf"
        else:
            prediction = model(torch.cat([zb, xb], dim=1))
            residual = (ab == 1).float() * prediction - 1.0
            critic = torch.cat([wb, xb], dim=1)
            kernel = "linear"
        loss = conditional_moment_loss(
            residual,
            critic,
            kernel,
            config.stabilized,
            config.lambda_stab,
            config.gamma_stab,
        )
        if kind == "q" and ablation:
            normalization_weight = (
                config.q_ablation_normalization_weight_sta
                if config.stabilized
                else config.q_ablation_normalization_weight_nonsta
            )
            loss = loss + normalization_weight * residual.mean().square()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
    return model.eval()


def fit_standard_nuisance(config, x, a, y):
    d, n = config.dimension, x.shape[0]
    outcome = OutcomeNet(d)
    propensity = PropensityNet(d)
    opt_m = torch.optim.RMSprop(outcome.parameters(), lr=config.learning_rate, momentum=config.momentum)
    opt_e = torch.optim.RMSprop(propensity.parameters(), lr=config.learning_rate, momentum=config.momentum)
    y_mean = y.mean()
    y_scale = y.std().clamp(min=1e-6)
    y_train = (y - y_mean) / y_scale
    batch = max(8, n // 40)
    for _ in range(config.updates):
        index = torch.randint(0, n, (batch,))
        xb, ab, yb = x[index], a[index], y_train[index]
        pred_y = outcome(torch.cat([ab, xb], dim=1))
        pred_a = propensity(xb)
        loss_y = F.mse_loss(pred_y, yb)
        loss_a = F.binary_cross_entropy(pred_a, ab)
        opt_m.zero_grad(); loss_y.backward(); opt_m.step()
        opt_e.zero_grad(); loss_a.backward(); opt_e.step()
    return ScaledOutcome(outcome.eval(), y_mean, y_scale).eval(), propensity.eval()


def linear_closed_estimate(x, z, w, a, y) -> float:
    # Keep the float32 pseudoinverse used by the reported implementation.  Its
    # numerical rank tolerance is important in the 121-dimensional design.
    phi = torch.cat([z, a, x], dim=1)
    psi = torch.cat([w, a, x], dim=1)
    tpsi = torch.cat([w, torch.ones_like(a), x], dim=1)
    moment = phi.T @ psi / len(x)
    target = (phi * y).mean(dim=0, keepdim=True).T
    return float(tpsi.mean(dim=0, keepdim=True) @ torch.linalg.pinv(moment) @ target)


def estimate_once(config: SyntheticConfig, rng: np.random.Generator, ablation: bool):
    x, z, w, a, y, truth = generate_data(config, rng, ablation)
    h = fit_bridge(config, x, z, w, a, y, "h", ablation=ablation)
    q = fit_bridge(config, x, z, w, a, y, "q", ablation=ablation)
    standard_m, standard_e = fit_standard_nuisance(config, x, a, y)
    with torch.no_grad():
        h1 = h(torch.cat([w, x], dim=1))
        q1 = q(torch.cat([z, x], dim=1)).clamp(max=50.0)
        reg = float(h1.mean())
        ipw = float((a * q1 * y).mean())
        dr = float((h1 + a * q1 * (y - h1)).mean())
        m1 = standard_m(torch.cat([torch.ones_like(a), x], dim=1))
        ma = standard_m(torch.cat([a, x], dim=1))
        ex = standard_e(x).clamp(0.01, 0.99)
        plain = float(m1.mean())
        standard_dr = float((m1 + a * (y - ma) / ex).mean())
    linear = linear_closed_estimate(x, z, w, a, y)
    return np.array([reg, ipw, dr, linear, plain, standard_dr], dtype=np.float64), truth


def summarize(estimates: np.ndarray, truth: float):
    relative = (estimates - truth) / truth
    mse = np.mean(relative**2, axis=0)
    bias2 = np.mean(relative, axis=0) ** 2
    variance = np.mean((relative - np.mean(relative, axis=0)) ** 2, axis=0)
    return np.stack([mse, bias2, variance], axis=1)


def run_block(config: SyntheticConfig, ablation: bool):
    seed = config.seed + (100_000 if ablation else 0)
    rng = np.random.default_rng(seed)
    estimates = np.empty((config.repetitions, 6), dtype=np.float64)
    truth = np.nan
    for repetition in range(config.repetitions):
        set_seed(seed + repetition)
        estimates[repetition], truth = estimate_once(config, rng, ablation)
    return estimates, float(truth), summarize(estimates, float(truth))


def format_summary(config, main_summary, ablation_summary, main_truth, ablation_truth):
    names = ["REG", "IPW", "DR", "Linear-closed", "Plain REG", "DR (no W,Z)"]
    lines = [
        f"config={json.dumps(asdict(config), sort_keys=True)}",
        f"main truth={main_truth:.8f}",
        "=== main: normalized MSE / bias^2 / variance ===",
    ]
    for name, values in zip(names, main_summary):
        lines.append(f"{name:<15}: NMSE={values[0]:.6f}, bias^2={values[1]:.6f}, var={values[2]:.6f}")
    lines.extend([
        "",
        f"ablation truth={ablation_truth:.8f}",
        "=== ablation: normalized MSE / bias^2 / variance ===",
    ])
    for name, values in zip(names, ablation_summary):
        lines.append(f"{name:<15}: NMSE={values[0]:.6f}, bias^2={values[1]:.6f}, var={values[2]:.6f}")
    return "\n".join(lines) + "\n"

In [2]:
TABLE_ID = 2
N = 400
TRANSFORM = 'sin'
ABLATION = False
REPETITIONS = 1000
# One sample-size-aware early-stopping rule is used in Tables 2--5.
# With batch=n//40, n=1200 still processes more examples in total.
UPDATES = 800 if N == 400 else 600

def run_variant(stabilized):
    config = SyntheticConfig(
        n=N, transform=TRANSFORM, stabilized=stabilized,
        repetitions=REPETITIONS, updates=UPDATES,
    )
    estimates, truth, summary_by_method = run_block(config, ablation=ABLATION)
    # summarize() returns method x metric; transpose to the manuscript's
    # metric x method layout used below.
    return estimates, truth, summary_by_method.T

non_estimates, non_truth, non_summary = run_variant(False)
sta_estimates, sta_truth, sta_summary = run_variant(True)
assert np.isclose(non_truth, sta_truth)

# run_block columns are REG, IPW, DR, Linear, Plain REG, DR(no W,Z),
# while the paper orders non-stabilized and stabilized bridge estimators in pairs.
corrected = np.column_stack([
    non_summary[:, 1], sta_summary[:, 1],
    non_summary[:, 0], sta_summary[:, 0],
    non_summary[:, 2], sta_summary[:, 2],
    non_summary[:, 3], non_summary[:, 4], non_summary[:, 5],
])


In [3]:
from IPython.display import Markdown, display
import pandas as pd

METHODS = [
    "IPW", "IPW(sta)", "REG", "REG(sta)", "DR", "DR(sta)",
    "Linear", "Plain REG", "DR (no W,Z)",
]
METRICS = ["MSE", "Squared Bias", "Variance"]
corrected_df = pd.DataFrame(corrected, index=METRICS, columns=METHODS)

# Validate only the freshly computed result; no cached result enters this notebook.
decomposition_error = float(np.max(np.abs(corrected[0] - corrected[1] - corrected[2])))
if decomposition_error > 1e-10 or not np.all(np.isfinite(corrected)) or np.any(corrected < 0):
    raise AssertionError("MSE/bias/variance integrity check failed")
if TABLE_ID <= 5 and not max(corrected[0, 4:6]) < min(corrected[0, 7:9]):
    raise AssertionError("Negative-control DR no longer outperforms methods ignoring W,Z")
if TABLE_ID in (6, 7) and not max(corrected[0, 0:2]) < 0.05:
    raise AssertionError("Ablation IPW is no longer competitive")

display(Markdown("## Corrected end-to-end full rerun"))
display(corrected_df.style.format(precision=4).set_caption(
    f"Table {TABLE_ID} — corrected full rerun"
))
print(f"VERIFIED: fresh-run MSE decomposition error={decomposition_error:.2e}; "
      "all qualitative checks passed.")

## Corrected end-to-end full rerun

,IPW,IPW(sta),REG,REG(sta),DR,DR(sta),Linear,Plain REG,"DR (no W,Z)"
MSE,0.0399,0.0353,0.0798,0.0892,0.0771,0.0844,0.1348,0.7718,0.7771
Squared Bias,0.0005,0.0322,0.0693,0.0794,0.0735,0.0800,0.1289,0.7660,0.7732
Variance,0.0394,0.0031,0.0105,0.0098,0.0037,0.0044,0.0059,0.0059,0.0039


VERIFIED: fresh-run MSE decomposition error=2.20e-15; all qualitative checks passed.
